# Dense vs. MoE serving benchmark
Run this notebook on one NVIDIA **L4 or H100** runtime. The full experiment starts one vLLM server at a time, records raw repetitions, and generates the two report figures. Do not use a T4 for the final FP8 measurements.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('Compute capability:', torch.cuda.get_device_capability(0))

In [ ]:
from pathlib import Path
REPO = 'Dense-and-Mixture-of-Experts-vLLM'
REPO_URL = 'https://github.com/itisaby/Dense-and-Mixture-of-Experts-vLLM.git'
if not Path('/content', REPO).exists():
    !git clone {REPO_URL} /content/{REPO}
%cd /content/{REPO}

In [ ]:
%pip install -q -r requirements.txt

## Run the experiment
The default is the full experiment. A quick plumbing check can use `--prefill-lengths 128 --decode-concurrencies 1 --repetitions 1`, but those smoke-test numbers are not suitable for the report.

In [ ]:
!python scripts/benchmark_serving.py --models all --output results/serving_raw.csv --overwrite

In [ ]:
!python scripts/plot_serving_results.py --input results/serving_raw.csv

In [ ]:
import pandas as pd
from IPython.display import Image, display
raw = pd.read_csv('results/serving_raw.csv')
summary = pd.read_csv('results/serving_summary.csv')
display(raw)
display(summary)
display(Image('figures/prefill_throughput.png'))
display(Image('figures/decode_throughput.png'))

In [ ]:
from google.colab import files
!zip -r benchmark_artifacts.zip results figures -x '*/README.md'
files.download('benchmark_artifacts.zip')